# 05 - Diagnóstico y optimización de un pipeline

## Conexión con la teoría

Partiremos de un pipeline funcional pero mejorable. Usaremos `explain()` y Spark UI para
formular hipótesis, cambiaremos una sola cosa cada vez y volveremos a observar.

In [ ]:
import os, shutil, subprocess

os.environ["JAVA_HOME"] = "/Users/beatrizsuch/java/temurin-17/Contents/Home"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("pontia-computacion-distribuida")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.master}")
print(f"Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## 1. Preparar una tabla de hechos y una dimensión

In [ ]:
transactions = (
    spark.range(0, 200_000, numPartitions=8)
    .withColumn("customer_id", (F.col("id") % 10_000).cast("long"))
    .withColumn("country_id", (F.col("id") % 20).cast("long"))
    .withColumn("amount", ((F.col("id") % 100) + 1).cast("double"))
)

countries = spark.createDataFrame(
    [(i, f"country_{i}", f"region_{i % 4}") for i in range(20)],
    ["country_id", "country", "region"],
)

## 2. Pipeline deliberadamente mejorable

In [ ]:
from pyspark.sql.types import DoubleType

@F.udf(DoubleType())
def add_tax_python(amount):
    return None if amount is None else amount * 1.21

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

baseline = (
    transactions
    .repartition(20)
    .withColumn("amount_with_tax", add_tax_python("amount"))
    .join(countries, "country_id")
    .groupBy("region")
    .agg(F.sum("amount_with_tax").alias("revenue"))
    .coalesce(1)
)

baseline.explain(mode="formatted")

### Formula hipótesis antes de cambiar nada

- ¿Era necesario `repartition(20)`?
- ¿Qué implica `BatchEvalPython` o `PythonUDF`?
- ¿Por qué aparece un sort-merge join con una dimensión tan pequeña?
- ¿Qué consecuencia tiene `coalesce(1)` para la salida?

## 3. Primera mejora: expresiones nativas y filtro temprano

In [ ]:
native = (
    transactions
    .filter(F.col("amount") > 0)
    .select("country_id", "amount")
    .withColumn("amount_with_tax", F.col("amount") * F.lit(1.21))
)
native.explain(mode="formatted")

Las expresiones nativas son visibles para Catalyst y evitan cruzar la frontera JVM-Python fila
a fila. Las UDF son útiles cuando no existe una expresión equivalente, no como primera opción.

## 4. Segunda mejora: broadcast de la dimensión

In [ ]:
optimized = (
    native
    .join(F.broadcast(countries), "country_id")
    .groupBy("region")
    .agg(F.round(F.sum("amount_with_tax"), 2).alias("revenue"))
)

optimized.explain(mode="formatted")
optimized.orderBy("region").show()

## 5. Cache solo si reutilizamos

In [ ]:
reusable = native.cache()
reusable.count()  # materialización

reusable.groupBy("country_id").count().show(5)
reusable.agg(F.avg("amount_with_tax")).show()

reusable.unpersist()

Comprueba en Spark UI si las acciones posteriores leen un bloque cacheado. Si el DataFrame se
utilizara una sola vez, el coste de materializarlo podría no compensar.

## 6. Checklist de Spark UI

Para cada ejecución revisa:

1. **Jobs:** cuántos ha creado cada acción.
2. **Stages:** dónde aparecen las fronteras de shuffle.
3. **Tasks:** si sus duraciones son equilibradas.
4. **Shuffle read/write:** cuánto dato se ha redistribuido.
5. **Spill:** si los datos intermedios han terminado en disco.
6. **Executors:** paralelismo, memoria y tareas fallidas.

La UI aporta síntomas; el plan permite formular una explicación.

## Preguntas de cierre

1. ¿Qué cambios eliminan movimiento de datos y cuáles eliminan coste de Python?
2. ¿Qué `Exchange` sigue siendo necesario para la agregación final?
3. ¿Por qué no debemos evaluar una optimización cambiando cinco cosas simultáneamente?
4. ¿Qué mejora esperarías que tuviera mayor impacto con datos realmente grandes?

In [ ]:
spark.stop()